In [1]:
# Генерация истинных наборов данных (ИНД) для каждого документа

# Для каждого документа мы:
# - Преобразуем документ в формат JSON, чтобы мы могли 
# отправить его в LLM.
# - Попросим LLM вернуть Questions - объект.
# - Создадим один ИНД для каждого сгенерированного вопроса

# Стоит учесть, что когда мы отправляем много запросов, 1 из них 
# может завершиться неудачей. Мы не хотим, чтобы вся партия 
# запросов завершилась неудачей из-за одной временной ошибки.

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data

documents = load_faq_data()

In [4]:
documents_llm = []

for doc in documents:
  if doc["course"] == "llm-zoomcamp":
    documents_llm.append(doc)

len(documents_llm)

153

In [5]:
documents = documents_llm

In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
  questions: list[str]

In [7]:
data_gen_instructions = """
  You emulate a student who's taking our course.
  Formulate 5 questions this student might ask based on a FAQ record. The record
  should contain the answer to the questions, and the questions should be complete and not too short.
  If possible, use as fewer words as possible from the record.

  The output should resemble how people ask questions
  on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from evaluation_utils import llm_structured_retry
import json

def generate_ground_truth(doc):
  # Преобразует объект в Python в json
  user_prompt = json.dumps(doc)

  out, usage = llm_structured_retry(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
  )

  results = []

  for q in out.questions:
    results.append({
      "question": q,
      "document": doc["id"]
    })

  return results, usage

In [9]:
generate_ground_truth(doc)

([{'question': 'How are capstone homework points usually counted, and what gives the most points?',
   'document': '0d200c8c58'},
  {'question': 'What do I need to do to get full score on a homework assignment?',
   'document': '0d200c8c58'},
  {'question': 'Is there a simple breakdown of how many points each homework task is worth?',
   'document': '0d200c8c58'},
  {'question': 'How many points can I earn in total from one homework, and what activities count toward it?',
   'document': '0d200c8c58'},
  {'question': 'Does the homework score depend on answering questions, sharing learning items, or adding FAQ questions?',
   'document': '0d200c8c58'}],
 ResponseUsage(input_tokens=268, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=367))

In [10]:
# Способ №1 - Добавляет для первых пяти документов вопросы
# Он будет выполнять 1 вызов LLM за другим. Запуск его для
# всех документов таким образом займет слишком много времени.

In [11]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
  records, usage = generate_ground_truth(doc)
  ground_truth.extend(records)
  usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
# ground_truth
'''
[{'question': 'Where can I find all the course materials and week folders for this class?',
  'document': '4487db3924'},
 {'question': 'Is there a specific GitHub folder for the current week’s content and homework?',
  'document': '4487db3924'},
 {'question': 'Where are the pre-recorded lectures posted, and how do I know if there are new workshops or updated videos?',
  'document': '4487db3924'},
 {'question': 'How do homework deadlines work, and what happens after the due date?',
  'document': '4487db3924'},
 {'question': 'Can I get extra points somehow, like by sharing my progress publicly on social media?',
  'document': '4487db3924'},
'''

usages
'''
[
  ResponseUsage(input_tokens=532, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=633),
  ResponseUsage(input_tokens=259, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=349),
  ResponseUsage(input_tokens=201, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=70, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=271),
  ResponseUsage(input_tokens=327, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=87, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=414),
  ResponseUsage(input_tokens=440, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=96, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=536)]
'''


'\n[\n  ResponseUsage(input_tokens=532, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=633),\n  ResponseUsage(input_tokens=259, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=349),\n  ResponseUsage(input_tokens=201, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=70, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=271),\n  ResponseUsage(input_tokens=327, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=87, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=414),\n  ResponseUsage(input_tokens=440, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=96, o

In [13]:
# Способ №2 - мы можем отправлять несколько запросов одновременно 
# и ждать их выполнения вместе. Мы обрабатываем документы параллельно
# и отслеживаем прогресс во время выполнения запросов. Однако лучше в 
# начале ограничиваться 5 соединениями

In [14]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [15]:
with ThreadPoolExecutor(max_workers = 5) as pool:
  results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

In [16]:
ground_truth = []
usages = []

for records, usage in results:
  ground_truth.extend(records)
  usages.append(usage)

len(ground_truth)

765

In [17]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
  cost = calc_price(usage)
  total_cost = total_cost + cost["total_cost"]

total_cost

0.11939399999999999

In [18]:
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

'''
  {
    'input_cost': 0.00015900000000000002,
    'output_cost': 0.00037799999999999997,
    'total_cost': 0.0005369999999999999
  }
'''

"\n  {\n    'input_cost': 0.00015900000000000002,\n    'output_cost': 0.00037799999999999997,\n    'total_cost': 0.0005369999999999999\n  }\n"

In [19]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.11939399999999999

In [20]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth

,question,document
0,I found this course late — can I still enroll ...,74eb249bbf
1,"Is it too late to join the course now, or can ...",74eb249bbf
2,Can I still participate in the course even if ...,74eb249bbf
3,"If I join late, do I still have a chance to ge...",74eb249bbf
4,What’s the deadline if I want a certificate af...,74eb249bbf
...,...,...
760,How do I switch the dlt workshop agent from Op...,54b98d3581
761,What should I change in the PydanticAI model s...,54b98d3581
762,Which environment variable do I need to add fo...,54b98d3581
763,"Do I need to modify `SearchDeps`, the instruct...",54b98d3581


In [21]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)